# SwiftSQL V1: 多源公开数据构建 + Unsloth QLoRA 微调 + GGUF 导出

> **Colab Pro 省积分建议**：请在右上角切换运行时为 **L4 GPU (24GB)**（仅 ~2 CU/小时），本 Notebook 完整跑完约 25–45 分钟（耗费约 1–2 积分）。

**本 Notebook 核心能力**：
1. 自动从 Hugging Face 拉取 3 个免登录公开数据集（`gretelai/synthetic_text_to_sql` + `trl-lab/SQaLe` + `CSpider/DuSQL` 中文）；
2. 用 `swiftsql` 流水线合成 **Auto-Think 自适应短推理 + PostgreSQL/MySQL/SQLite 三方言 + 20% 报错自修复 + 10% 缺列拒答** 语料；
3. 用 Unsloth 在 `Qwen/Qwen2.5-Coder-3B-Instruct`（可一键改 `7B`）上跑完 V1 SFT，并将 Checkpoint 和 **`Q8_0` / `Q4_K_M` GGUF** 持久化到 Google Drive。


In [ ]:
# 1. 检查 GPU 并挂载 Google Drive（所有数据与权重都持久化到 Drive）
import os, torch
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/swift-sql"
os.makedirs(DRIVE_DIR, exist_ok=True)

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Ready: {name} ({vram:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("未检测到 GPU！请在 Colab 菜单：代码执行程序 -> 更改运行时类型 -> 选择 L4 GPU。")


In [ ]:
# 2. 拉取 GitHub 仓库并安装依赖
REPO_URL = "https://github.com/2422725977/swift-sql.git"
REPO_DIR = "/content/swift-sql"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q --upgrade unsloth trl datasets sqlglot pyyaml
!pip install -q --no-deps -e {REPO_DIR}
import sys
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")


In [ ]:
# 3. 构建 V1 多任务训练集与独立验证集（并缓存到 Google Drive）
from collections import Counter
from pathlib import Path
import yaml

from swiftsql.data_pipeline import (
    build_sft_records,
    load_jsonl,
    load_public_hf_datasets,
    save_jsonl,
)

with open(f"{REPO_DIR}/configs/train_config.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

DATA_DIR = Path(DRIVE_DIR) / "data"
TRAIN_PATH = DATA_DIR / "train_sft.jsonl"
EVAL_PATH = DATA_DIR / "eval.jsonl"

FORCE_REBUILD = False  # 设为 True 可强制重新生成数据

if TRAIN_PATH.exists() and EVAL_PATH.exists() and not FORCE_REBUILD:
    train_records = load_jsonl(TRAIN_PATH)
    eval_records = load_jsonl(EVAL_PATH)
    print(f"Loaded cached datasets from Drive: train={len(train_records)}, eval={len(eval_records)}")
else:
    dcfg = CFG["data"]
    print("Downloading open Hugging Face datasets (Gretel + SQaLe + CSpider/DuSQL)...")
    train_raw, eval_raw = load_public_hf_datasets(
        gretel_train_samples=dcfg["gretel_train_samples"],
        sqale_train_samples=dcfg["sqale_train_samples"],
        chinese_train_samples=dcfg["chinese_train_samples"],
        eval_samples=dcfg["eval_samples"],
        seed=CFG["project"]["seed"],
    )
    train_records = build_sft_records(
        train_raw,
        dialect_weights=dcfg["dialect_weights"],
        repair_ratio=dcfg["repair_ratio"],
        refusal_ratio=dcfg["refusal_ratio"],
        max_schema_chars=dcfg["max_schema_chars"],
        seed=CFG["project"]["seed"],
    )
    eval_records = build_sft_records(
        eval_raw,
        dialect_weights=dcfg["dialect_weights"],
        repair_ratio=0.10,
        refusal_ratio=0.15,
        max_schema_chars=dcfg["max_schema_chars"],
        seed=CFG["project"]["seed"] + 1,
    )
    save_jsonl(train_records, TRAIN_PATH)
    save_jsonl(eval_records, EVAL_PATH)
    print(f"Saved to Drive: train={len(train_records)}, eval={len(eval_records)}")

print("Train task mix:", Counter(r["task_type"] for r in train_records))
print("Train dialect mix:", Counter(r["dialect"] for r in train_records))
print("Train complexity mix:", Counter(r["complexity"] for r in train_records))
print("Train language mix:", Counter(r["language"] for r in train_records))


In [ ]:
# 4. 加载基座模型并挂载 LoRA（默认 3B，最适合 Mac mini 16GB Q8_0 极速高并发）
from unsloth import FastLanguageModel

BASE_MODEL = CFG["model"]["base_model"]  # 如需尝试 7B，可改为 "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = CFG["model"]["max_seq_length"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=CFG["model"]["load_in_4bit"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora"]["r"],
    target_modules=CFG["lora"]["target_modules"],
    lora_alpha=CFG["lora"]["lora_alpha"],
    lora_dropout=CFG["lora"]["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CFG["project"]["seed"],
)
model.print_trainable_parameters()


In [ ]:
# 5. 格式化 ChatML 并启动 V1 SFT 训练（自动存 Checkpoint 到 Google Drive）
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_text(records):
    texts = []
    for r in records:
        text = tokenizer.apply_chat_template(
            r["messages"], tokenize=False, add_generation_prompt=False
        )
        texts.append({"text": text})
    return Dataset.from_list(texts)

train_ds = to_text(train_records)
eval_ds = to_text(eval_records[:200])

sft_cfg = CFG["sft"]
OUT_DIR = f"{DRIVE_DIR}/checkpoints/v1_sft"
use_bf16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir=OUT_DIR,
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        num_train_epochs=sft_cfg["num_train_epochs"],
        per_device_train_batch_size=sft_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=sft_cfg["gradient_accumulation_steps"],
        learning_rate=sft_cfg["learning_rate"],
        warmup_ratio=sft_cfg["warmup_ratio"],
        lr_scheduler_type=sft_cfg["lr_scheduler_type"],
        logging_steps=sft_cfg["logging_steps"],
        save_steps=sft_cfg["save_steps"],
        eval_strategy="steps",
        eval_steps=sft_cfg["eval_steps"],
        save_total_limit=3,
        fp16=not use_bf16,
        bf16=use_bf16,
        optim="adamw_8bit",
        seed=CFG["project"]["seed"],
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("V1 SFT finished! Runtime (s):", round(trainer_stats.metrics["train_runtime"], 1))

LORA_DIR = f"{DRIVE_DIR}/models/v1_sft_lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print("Saved V1 LoRA adapter to:", LORA_DIR)


In [ ]:
# 6. 四合一快速人工验收：简单直出 / 复杂 Auto-Think / 缺列拒答 / 报错自修复
import time
from swiftsql.prompts import build_generation_messages, build_repair_messages

FastLanguageModel.for_inference(model)

def ask(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0)
    ms = (time.perf_counter() - t0) * 1000
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen.strip(), round(ms, 1)

schema = (
    "CREATE TABLE users (id INT, name TEXT, city TEXT);\n"
    "CREATE TABLE orders (id INT, user_id INT, amount NUMERIC, status TEXT);"
)

tests = [
    ("1. 简单单表(应快且plan为空)", build_generation_messages(schema, "查询所有北京用户的姓名", dialect="postgresql")),
    ("2. 复杂JOIN聚合(应含1句plan)", build_generation_messages(schema, "统计每个城市已完成订单的总金额，取前3名", dialect="mysql")),
    ("3. 缺列拒答(应answerable=false)", build_generation_messages(schema, "查询所有用户的手机号和邮箱", dialect="postgresql")),
    ("4. 报错自修复(repair_sql)", build_repair_messages(schema, "统计每个用户的订单数", "SELECT user_id, COUNT(*) FORM orders GROUP BY user_id", 'syntax error at or near "FORM"', dialect="postgresql")),
]

for title, msgs in tests:
    ans, ms = ask(msgs)
    print(f"=== {title} ({ms} ms) ===")
    print(ans, "\n")


In [ ]:
# 7. 导出 GGUF 到 Google Drive（直接下载到 Mac mini 16GB 运行）
GGUF_DIR = f"{DRIVE_DIR}/exports/v1_sft_gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

for q in CFG["export"]["quantizations"]:
    print(f"Exporting GGUF ({q}) -> {GGUF_DIR} ...")
    model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=q)

!ls -lh {GGUF_DIR}
